In [1]:
import pandas as pd
import numpy as np

ipl = pd.read_csv("ipl_cleaned_data.csv")


/tmp/ipython-input-2923981201.py:4: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  ipl = pd.read_csv("ipl_cleaned_data.csv")


In [2]:
batsman_match = (
    ipl.groupby(["matchid", "batsman", "batting_team", "venue", "season"])
    .agg(
        runs_scored=("batsman_runs", "sum"),
        balls_faced=("ball", "count")
    )
    .reset_index()
)


In [3]:
bowler_match = (
    ipl.groupby(["matchid", "bowler", "bowling_team", "venue", "season"])
    .agg(
        wickets_taken=("player_dismissed", lambda x: (x != "Not Out").sum()),
        runs_conceded=("total_runs", "sum"),
        balls_bowled=("ball", "count")
    )
    .reset_index()
)

bowler_match["overs_bowled"] = bowler_match["balls_bowled"] / 6


In [4]:
batsman_match["strike_rate"] = np.where(
    batsman_match["balls_faced"] > 0,
    (batsman_match["runs_scored"] / batsman_match["balls_faced"]) * 100,
    0
)

bowler_match["economy"] = np.where(
    bowler_match["overs_bowled"] > 0,
    bowler_match["runs_conceded"] / bowler_match["overs_bowled"],
    0
)


In [5]:
batsman_match = batsman_match.sort_values(["batsman", "matchid"])

batsman_match["avg_runs_last_5"] = (
    batsman_match
    .groupby("batsman")["runs_scored"]
    .rolling(5, min_periods=1)
    .mean()
    .reset_index(level=0, drop=True)
)


In [6]:
bowler_match = bowler_match.sort_values(["bowler", "matchid"])

bowler_match["avg_wickets_last_5"] = (
    bowler_match
    .groupby("bowler")["wickets_taken"]
    .rolling(5, min_periods=1)
    .mean()
    .reset_index(level=0, drop=True)
)


In [7]:
batsman_match["venue_avg_runs"] = (
    batsman_match
    .groupby(["batsman", "venue"])["runs_scored"]
    .transform("mean")
)

bowler_match["venue_avg_wickets"] = (
    bowler_match
    .groupby(["bowler", "venue"])["wickets_taken"]
    .transform("mean")
)


In [8]:
batsman_match["avg_runs_vs_team"] = (
    batsman_match
    .groupby(["batsman", "batting_team"])["runs_scored"]
    .transform("mean")
)


In [9]:
bowler_match["avg_wickets_vs_team"] = (
    bowler_match
    .groupby(["bowler", "bowling_team"])["wickets_taken"]
    .transform("mean")
)


In [10]:
batsman_match.to_csv("batsmen_features.csv", index=False)
bowler_match.to_csv("bowlers_features.csv", index=False)
